In [13]:
import numpy as np
import os

def generate_2d_riemann_mesh(
    nx=1000,
    ny=1000,
    gamma=1.4,
    output_dir="./mesh"
):
    os.makedirs(output_dir, exist_ok=True)

    # 计算域 [0,1]x[0,1]
    domain_size = 1.0
    da = domain_size / nx

    rho    = np.zeros((ny, nx))
    u      = np.zeros((ny, nx))
    v      = np.zeros((ny, nx))
    p      = np.zeros((ny, nx))
    bctype = np.zeros((ny, nx), dtype=np.int32)

    # 界面位置（列/行索引）
    # i=0 对应 y=1（顶部），i 增大 y 减小
    # j=0 对应 x=0（左侧），j 增大 x 增大
    x_int = 0.8   # x 界面
    y_int = 0.8   # y 界面

    for i in range(ny):
        for j in range(nx):
            # 单元中心坐标
            xc = (j + 0.5) * da          # x ∈ [0,1]
            yc = 1.0 - (i + 0.5) * da   # y ∈ [0,1]，i=0→顶部

            if xc >= x_int and yc >= y_int:
                # 右上象限：ρ=1.5, u=0, v=0, p=1.5
                rho[i,j], u[i,j], v[i,j], p[i,j] = 1.5, 0.0, 0.0, 1.5

            elif xc < x_int and yc >= y_int:
                # 左上象限：ρ=0.5323, u=1.206, v=0, p=0.3
                rho[i,j], u[i,j], v[i,j], p[i,j] = 0.5323, 1.206, 0.0, 0.3

            elif xc < x_int and yc < y_int:
                # 左下象限：ρ=0.138, u=1.206, v=1.206, p=0.029
                rho[i,j], u[i,j], v[i,j], p[i,j] = 0.138, 1.206, -1.206, 0.029

            else:
                # 右下象限：ρ=0.5323, u=0, v=1.206, p=0.3
                rho[i,j], u[i,j], v[i,j], p[i,j] = 0.5323, 0.0, -1.206, 0.3

            # 边界类型：最外两圈为 -1（固壁/零梯度）
            if i < 2 or i >= ny-2 or j < 2 or j >= nx-2:
                bctype[i,j] = -1
            else:
                bctype[i,j] = 0

    # 保存 params.txt
    with open(os.path.join(output_dir, "params.txt"), "w") as f:
        f.write(f"{nx} {ny} {da:.10e} {gamma}\n")

    # 保存矩阵（Eigen 行优先文本格式）
    def save_matrix(filename, data):
        with open(os.path.join(output_dir, filename), "w") as f:
            for i in range(data.shape[0]):
                for j in range(data.shape[1]):
                    f.write(f"{data[i,j]:.10e} ")
                f.write("\n")

    save_matrix("rho.dat",    rho)
    save_matrix("u.dat",      u)
    save_matrix("v.dat",      v)
    save_matrix("p.dat",      p)
    save_matrix("bctype.dat", bctype.astype(float))

    print("========== 四象限初始条件 ==========")
    print(f"右上 (x≥0.8, y≥0.8): ρ=1.5,    u=0,     v=0,     p=1.5")
    print(f"左上 (x<0.8, y≥0.8): ρ=0.5323, u=1.206, v=0,     p=0.3")
    print(f"左下 (x<0.8, y<0.8): ρ=0.138,  u=1.206, v=1.206, p=0.029")
    print(f"右下 (x≥0.8, y<0.8): ρ=0.5323, u=0,     v=1.206, p=0.3")
    print(f"\n网格: {ny}×{nx}, da={da:.6f}, 输出: {os.path.abspath(output_dir)}")

if __name__ == "__main__":
    generate_2d_riemann_mesh(nx=1000, ny=1000, gamma=1.4, output_dir="./mesh")

========== 四象限初始条件 ==========
右上 (x≥0.8, y≥0.8): ρ=1.5,    u=0,     v=0,     p=1.5
左上 (x<0.8, y≥0.8): ρ=0.5323, u=1.206, v=0,     p=0.3
左下 (x<0.8, y<0.8): ρ=0.138,  u=1.206, v=1.206, p=0.029
右下 (x≥0.8, y<0.8): ρ=0.5323, u=0,     v=1.206, p=0.3

网格: 1000×1000, da=0.001000, 输出: /home/midway/ZuikakuCFD/mesh
